In [28]:
import pandas as pd
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

In [29]:
df = pd.read_csv('dataset/train.csv')
df

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy
...,...,...,...,...,...,...,...,...,...,...,...
77294,77294,qp0d4n,49,2:0,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy
77295,77295,qp0d4q,49,2:0,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy
77296,77296,qp0d4w,49,2:0,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny
77297,77297,qp0dhw,49,2:0,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny


In [30]:
df.shape

(77299, 11)

In [31]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month']       = df['timestamp'].dt.month
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df = df.drop(columns=['timestamp'])

/tmp/ipykernel_9780/720111181.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'])


In [32]:
df.isna().sum()

Index               0
geohash             0
day                 0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
hour                0
day_of_week         0
month               0
is_weekend          0
dtype: int64

In [33]:
df = df.dropna()
df = df.drop(columns=['Index'])
print(df.isna().sum())
print(df.shape)
df

geohash          0
day              0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
hour             0
day_of_week      0
month            0
is_weekend       0
dtype: int64
(73459, 13)


,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,day_of_week,month,is_weekend
1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,1,0
2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,1,0
4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,1,0
5,qp02zw,48,0.016262,Residential,2,Not Allowed,Yes,8.446025,Rainy,0,0,1,0
6,qp02zy,48,0.042247,Residential,3,Allowed,Yes,15.772408,Foggy,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
77294,qp0d4n,49,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy,2,0,1,0
77295,qp0d4q,49,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy,2,0,1,0
77296,qp0d4w,49,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny,2,0,1,0
77297,qp0dhw,49,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny,2,0,1,0


In [34]:
df['geohash'].unique()

<StringArray>
['qp02zt', 'qp08bj', 'qp02zq', 'qp02zw', 'qp02zy', 'qp08by', 'qp08gq',
 'qp08gy', 'qp02zp', 'qp02zr',
 ...
 'qp092z', 'qp09du', 'qp097m', 'qp09y4', 'qp03zy', 'qp03yn', 'qp09vs',
 'qp098v', 'qp0d5c', 'qp0d55']
Length: 1248, dtype: str

In [35]:
X = df.drop(columns=['demand'])  
y = df['demand']    

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [37]:
categorical_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', TargetEncoder(), categorical_cols)
    ],
    remainder='passthrough'
)

In [11]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist',
    device='cuda', 
    eval_metric='rmse',
    early_stopping_rounds=20,  
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

In [12]:
param_grid = {
    'model__n_estimators':      [200, 400, 800],
    'model__max_depth':         [4, 6, 8],
    'model__learning_rate':     [0.01, 0.05, 0.1],
    'model__subsample':         [0.8, 1.0],
    'model__colsample_bytree':  [0.7, 0.8, 1.0],
    'model__min_child_weight':  [1, 3, 5],
    'model__reg_alpha':         [0, 0.1],  
    'model__reg_lambda':        [1, 1.5],   
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    verbose=1,
    n_jobs=-1   
)

X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_test_preprocessed  = preprocessor.transform(X_test)

grid_search.fit(
    X_train, y_train,
    model__eval_set=[(X_test_preprocessed, y_test)], 
    model__verbose=False
)

print("Best Params:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_)

Fitting 5 folds for each of 1944 candidates, totalling 9720 fits


/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [14:50:59] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [14:51:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordin

Best Params: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.01, 'model__max_depth': 6, 'model__min_child_weight': 5, 'model__n_estimators': 800, 'model__reg_alpha': 0.1, 'model__reg_lambda': 1, 'model__subsample': 0.8}
Best CV R²: 0.91762229958166


In [13]:
y_pred = grid_search.best_estimator_.predict(X_test)
print("Test R²:", r2_score(y_test, y_pred))

Test R²: 0.9206271713946353


/home/akshat/gridlock/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [16:02:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [38]:
df_test    = pd.read_csv('dataset/test.csv')
test_index = df_test['Index']
df_test    = df_test.drop(columns=['Index'])
df_test

,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy
...,...,...,...,...,...,...,...,...,...
41773,qp0d4q,49,13:45,Street,1,Not Allowed,Yes,19.588991,Sunny
41774,qp0d4w,49,13:45,Residential,2,Not Allowed,Yes,10.735538,Rainy
41775,qp0dhq,49,13:45,Residential,2,Not Allowed,Yes,13.223750,Rainy
41776,qp0dhw,49,13:45,Residential,2,Not Allowed,Yes,12.510917,Rainy


In [39]:
df_test.isna().sum()

geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64

In [40]:
for col in ['RoadType', 'Weather']:
    df_test[col] = df_test[col].fillna(X_train[col].mode()[0])
df_test['Temperature'] = df_test['Temperature'].fillna(X_train['Temperature'].median())

df_test.isna().sum()

geohash          0
day              0
timestamp        0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64

In [41]:
df_test['timestamp']   = pd.to_datetime(df_test['timestamp'])
df_test['hour']        = df_test['timestamp'].dt.hour
df_test['day_of_week'] = df_test['timestamp'].dt.dayofweek
df_test['month']       = df_test['timestamp'].dt.month
df_test['is_weekend']  = df_test['day_of_week'].isin([5, 6]).astype(int)
df_test = df_test.drop(columns=['timestamp'])

df_X_test = df_test.copy()

/tmp/ipykernel_9780/22034135.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_test['timestamp']   = pd.to_datetime(df_test['timestamp'])


In [42]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist',
    device='cuda',    
    # ── Best params from GridSearch ──
    n_estimators=800,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=1.0,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

In [43]:
pipeline.fit(X_train, y_train)
print("Train R²:", r2_score(y_test, pipeline.predict(X_test)))

Train R²: 0.920368616508026


In [44]:
y_pred = pipeline.predict(df_X_test)

submission = pd.DataFrame({
    'Index':  test_index,
    'demand': y_pred
})

submission

,Index,demand
0,0,0.049438
1,1,0.044123
2,2,0.036790
3,3,0.049537
4,4,0.066863
...,...,...
41773,41773,0.272394
41774,41774,0.131313
41775,41775,0.018008
41776,41776,0.102478


In [45]:
submission.to_csv('submission.csv', index=False)